# HW5: Deep Neural Network from Scratch

MNIST digits 0 to 9. Network is 784 input, 100 hidden with Sigmoid, 10 output with Softmax.

Sigmoid, Softmax, cross entropy, the backward pass and gradient descent are all written from
scratch. No autograd, no optimizer, no DataLoader.

In [1]:
import csv
import torch

torch.manual_seed(42)
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

## Load data

First value in each row is the label, the other 784 are pixels. Pixels are scaled to [0, 1].
Labels are one hot encoded because the output layer has 10 neurons.

In [2]:
def load_csv(path):
    features, labels = [], []
    with open(path) as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            labels.append(int(row[0]))
            features.append([float(v) for v in row[1:]])
    return torch.tensor(features) / 255.0, torch.tensor(labels)


def one_hot(y):
    Y = torch.zeros(y.shape[0], 10)
    Y[torch.arange(y.shape[0]), y] = 1.0
    return Y


X_train, y_train = load_csv("mnist_train.csv")
X_test, y_test = load_csv("mnist_test.csv")
Y_train, Y_test = one_hot(y_train), one_hot(y_test)

print("train:", tuple(X_train.shape), "test:", tuple(X_test.shape))

train: (60000, 784) test: (10000, 784)


## 1. Build the network

$$\sigma(z) = \frac{1}{1 + \exp(-z)}
\qquad
a^{[2]}_i = \frac{\exp(z^{[2]}_i)}{\sum_{j=1}^{K}\exp(z^{[2]}_j)}$$

Softmax subtracts the row max so exp does not overflow. This cancels out and does not
change the result.

Forward pass:

$$Z^{[1]} = X W^{[1]} + b^{[1]} \qquad A^{[1]} = \sigma(Z^{[1]})$$
$$Z^{[2]} = A^{[1]} W^{[2]} + b^{[2]} \qquad A^{[2]} = \text{softmax}(Z^{[2]}) = \hat{y}$$

In [3]:
def sigmoid(z):
    return 1.0 / (1.0 + torch.exp(-z))


def softmax(z):
    e = torch.exp(z - z.max(dim=1, keepdim=True).values)
    return e / e.sum(dim=1, keepdim=True)


def initialize_parameters(n_in=784, n_h=100, n_out=10):
    return {
        "W1": torch.randn(n_in, n_h) * (1.0 / n_in) ** 0.5,
        "b1": torch.zeros(1, n_h),
        "W2": torch.randn(n_h, n_out) * (1.0 / n_h) ** 0.5,
        "b2": torch.zeros(1, n_out),
    }


def forward(X, params):
    A1 = sigmoid(X @ params["W1"] + params["b1"])
    A2 = softmax(A1 @ params["W2"] + params["b2"])
    return A2, {"X": X, "A1": A1, "A2": A2}

## 2. Loss, accuracy and backward propagation

$$J = -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=1}^{K} y^{i}_{k}\log(\hat{y}^{i}_{k})$$

Softmax with cross entropy makes the output gradient simply $A^{[2]} - Y$.

$$dZ^{[2]} = \frac{1}{m}(A^{[2]} - Y)
\qquad dW^{[2]} = A^{[1]T} dZ^{[2]}
\qquad db^{[2]} = \sum_i dZ^{[2]}_i$$

$$dZ^{[1]} = (dZ^{[2]} W^{[2]T}) \odot A^{[1]}(1 - A^{[1]})
\qquad dW^{[1]} = X^{T} dZ^{[1]}
\qquad db^{[1]} = \sum_i dZ^{[1]}_i$$

In [4]:
def cross_entropy(Y, A2):
    return -torch.mean(torch.sum(Y * torch.log(A2 + 1e-8), dim=1))


def accuracy(y, A2):
    return (A2.argmax(dim=1) == y).float().mean().item()


def backward(Y, cache, params):
    m = Y.shape[0]
    X, A1, A2 = cache["X"], cache["A1"], cache["A2"]

    dZ2 = (A2 - Y) / m
    dW2 = A1.t() @ dZ2
    db2 = dZ2.sum(dim=0, keepdim=True)

    dZ1 = (dZ2 @ params["W2"].t()) * A1 * (1 - A1)
    dW1 = X.t() @ dZ1
    db1 = dZ1.sum(dim=0, keepdim=True)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


def update_parameters(params, grads, learning_rate):
    for key in params:
        params[key] = params[key] - learning_rate * grads[key]
    return params

## 3. Training

Gradient descent written by hand. Batches come from torch.randperm and slicing, so no
DataLoader is used. Learning rate 0.5, batch size 128, 30 epochs.

In [5]:
learning_rate = 0.5
epochs = 30
batch_size = 128

params = initialize_parameters()
m = X_train.shape[0]

print(f"{'epoch':>5} {'loss':>10} {'train acc':>11} {'test acc':>10}")
for epoch in range(1, epochs + 1):
    permutation = torch.randperm(m)

    for start in range(0, m, batch_size):
        idx = permutation[start:start + batch_size]
        _, cache = forward(X_train[idx], params)
        grads = backward(Y_train[idx], cache, params)
        params = update_parameters(params, grads, learning_rate)

    A_train, _ = forward(X_train, params)
    A_test, _ = forward(X_test, params)
    train_acc = accuracy(y_train, A_train)
    test_acc = accuracy(y_test, A_test)

    print(f"{epoch:>5} {cross_entropy(Y_train, A_train):>10.4f} {train_acc:>11.4f} {test_acc:>10.4f}")

print()
print(f"final train accuracy: {train_acc:.4f}")
print(f"final test accuracy:  {test_acc:.4f}")
print(f"both above 0.91: {train_acc > 0.91 and test_acc > 0.91}")

epoch       loss   train acc   test acc
    1     0.3079      0.9116     0.9152


    2     0.2518      0.9269     0.9275
    3     0.2195      0.9363     0.9350


    4     0.1910      0.9441     0.9423
    5     0.1687      0.9510     0.9484


    6     0.1470      0.9581     0.9532
    7     0.1330      0.9620     0.9574


    8     0.1199      0.9654     0.9609
    9     0.1133      0.9676     0.9621


...

   22     0.0574      0.9838     0.9728
   23     0.0498      0.9868     0.9750


   24     0.0469      0.9880     0.9756
   25     0.0452      0.9888     0.9752


   26     0.0442      0.9886     0.9754
   27     0.0424      0.9895     0.9769


   28     0.0407      0.9901     0.9757
   29     0.0384      0.9907     0.9758


   30     0.0375      0.9912     0.9764

final train accuracy: 0.9912
final test accuracy:  0.9764
both above 0.91: True
